In [ ]:
import torch
# open file
# define vocabulary
# create encoder and decoder
# encode file as data through encoder
# split into train and val sets
with open('input.txt', 'r', encoding='utf-8') as f:
	data = f.read()

print("length of database:", len(data))
print(data[0:1000])

chars = sorted(list(set(data)))
vocab_size = len(chars)
print(vocab_size)
print("".join(chars))

stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(data), dtype=torch.long)
print(data.shape, data.dtype)
print(data[0:1000])
print(torch.cuda.is_available())

n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [15]:
# one example
block_size = 8
x = train_data[:block_size]
print(x)
y = train_data[1:block_size+1]
print(y)



tensor([18, 47, 56, 57, 58,  1, 15, 47])
tensor([47, 56, 57, 58,  1, 15, 47, 58])


In [16]:
torch.manual_seed(1337)

batch_size = 4

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data)-block_size, (batch_size, ))
    
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    
    return x, y

x, y = get_batch('train')
print("xshape:", x.shape, x, "\nyshape:", y.shape, y)


xshape: torch.Size([4, 8]) tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]]) 
yshape: torch.Size([4, 8]) tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


In [ ]:
import torch.nn as nn
from torch.nn import functional as F 
torch.manual_seed(1337)

class BigramLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, vocab_size)
        
    def forward(self, idx, targets):
        logits = self.wte(idx)
        B, T, C = logits.shape
        logits = logits.view(B*T, C)
        targets = targets.view(B*T)
        loss = F.cross_entropy(logits, targets)
        return logits, loss

m = BigramLM(vocab_size)
logits, loss = m(x, y)
print("shape", logits.shape)
print("\noutput", logits)
print("\n", loss)
# print("\ngurt: yo", decode(logits))


shape torch.Size([32, 65])

output tensor([[-1.5101, -0.0948,  1.0927,  ..., -0.6126, -0.6597,  0.7624],
        [ 0.3323, -0.0872, -0.7470,  ..., -0.6716, -0.9572, -0.9594],
        [ 0.2475, -0.6349, -1.2909,  ...,  1.3064, -0.2256, -1.8305],
        ...,
        [-2.1910, -0.7574,  1.9656,  ..., -0.3580,  0.8585, -0.6161],
        [ 0.5978, -0.0514, -0.0646,  ..., -1.4649, -2.0555,  1.8275],
        [-0.6787,  0.8662, -1.6433,  ...,  2.3671, -0.7775, -0.2586]],
       grad_fn=<ViewBackward0>)

 tensor(4.8786, grad_fn=<NllLossBackward0>)
